# Boston Metro Clean for Massachusetts Crash Map 
# (Expected Runtime: 5-8 minutes)

Output in /data folder:

* boston-metro-clean.csv - contains MassDOT crash records for specific region in proper format for crash map


# Importing all libraries

In [1]:
# !pip install googlemaps
# uncomment the line above if we decide to pay for GOOGLE CLOUD, an estimate of the cost is below $200
# GOOGLE CLOUD can be used to geocode 16911 crashes that have addresses but not lon and lat
# more information on how to do this can be found here: https://tinyurl.com/mr42hkxw
!pip install pandas
!pip install numpy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Collecting MassDOT Crash Records 2022-2027 for Regional Planning Area MAPC (Boston metro)

In [2]:
all_features = []

years = range(2022, 2027)

for year in years:
   
    base_url = f"https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT/MASSDOT_ODP_OPEN_{year}/FeatureServer/0/query"
    
    params = {
        "where": " RPA_ABBR = 'MAPC'",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "returnGeometry": "true",
        "resultOffset": 0,
        "resultRecordCount": 2000
    }

    while True:
        
        response = requests.get(base_url, params=params)
        data = response.json()
        features = data.get("features", [])
        
        if not features:
            break
        
        all_features.extend(features)
        params["resultOffset"] += params["resultRecordCount"]

records = [f["attributes"] for f in all_features]

df = pd.DataFrame(records)
print("Shape:", df.shape)
df.head()

Shape: (166191, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATE_TEXT,CRASH_TIME_2,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,5051236,EVERETT,01 01 2022,11:34 AM,1.641037e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No injury,2,...,NaN,Major Collector,4742067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5051675,EVERETT,01 03 2022,3:12 AM,1.641180e+12,03:00AM to 03:59AM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,...,NaN,Principal Arterial - Other,4742121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5051674,EVERETT,01 02 2022,10:17 PM,1.641162e+12,10:00PM to 10:59PM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,...,NaN,Local,4742122,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5051661,DEDHAM,01 02 2022,12:36 PM,1.641127e+12,12:00PM to 12:59PM,Closed,Unknown,Not reported,2,...,NaN,Local,4742135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5051578,BRAINTREE,01 02 2022,11:49 AM,1.641124e+12,11:00AM to 11:59AM,Closed,Non-fatal injury,Possible Injury (C),3,...,NaN,Minor Arterial,4742178,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
mass_crashes = df
print("Shape of MASSDOT dataset:", mass_crashes.shape)
mass_crashes.head()

Shape of MASSDOT dataset: (166191, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL,CRASH_TIME,CRASH_DATE
0,5051236,EVERETT,1.641037e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No injury,2,0,0,...,4742067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11:34:00,2022-01-01
1,5051675,EVERETT,1.641180e+12,03:00AM to 03:59AM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,4742121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03:12:00,2022-01-03
2,5051674,EVERETT,1.641162e+12,10:00PM to 10:59PM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,4742122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22:17:00,2022-01-02
3,5051661,DEDHAM,1.641127e+12,12:00PM to 12:59PM,Closed,Unknown,Not reported,2,0,0,...,4742135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12:36:00,2022-01-02
4,5051578,BRAINTREE,1.641124e+12,11:00AM to 11:59AM,Closed,Non-fatal injury,Possible Injury (C),3,6,0,...,4742178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11:49:00,2022-01-02


# Standardizing datasets before merging

**The following code creates new columns:**
* The CYCLIST column has a value of 1 if a cyclist was involved in a crash, 0 otherwise.
* The PEDESTRIAN column has a value of 1 if a pedestrian was involved in a crash, 0 otherwise.
* The OTHER column has a value of 1 if another vulnerable user was involved in a crash, 0 otherwise.

**Standardizing MASSDOT crashes**

**Terms identified as a Pedestrian:** 
* Pedestrian
* Non-Motorized Wheelchair User
* Roller Skater

**Terms identified as a Cyclist:** 
* Cyclist 
* Bicyclist
* Bicycle

**Terms identified as Other:** 
* moped
* other
* unknown
* skateboarder
* Motorized Scooter Rider
* passenger
* Emergency Responder
* Roadway Worker

In [4]:
col1 = mass_crashes["NON_MTRST_TYPE_CL"]
col2 = mass_crashes["MOST_HRMFL_EVT_CL"]

mass_crashes["CYCLIST"] = np.where(
    col1.str.contains("Cyclist|Bicyclist", case=False, na=False) |
    col2.str.contains("Cyclist|Bicyclist|Bicycle", case=False, na=False),
    1,
    0
)

mass_crashes["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Non-Motorized Wheelchair|Roller Skater", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

mass_crashes['INTERSTATE'] = np.where(
    mass_crashes['F_CLASS'].str.contains("Interstate", case=False, na=False),
    1,
    0
)

mass_crashes['SEVERITY'] = mass_crashes['CRASH_SEVERITY_DESCR'].map({
    'Fatal injury': 1,
    'Non-fatal injury': 2
}).fillna(0)

mass_crashes["OTHER"] = np.where(
    mass_crashes["NON_MTRST_TYPE_CL"].str.contains("moped|other|unknown|skateboarder|Motorized Scooter Rider|passenger|Emergency Responder|Roadway Worker", case=False, na=False) |
    mass_crashes["MOST_HRMFL_EVT_CL"].str.contains("moped|other vulnerable", case=False, na=False),
    1,
    0
)

mass_crashes['POLICE'] = mass_crashes['POLC_AGNCY_TYPE_DESCR']
mass_crashes['ID'] = mass_crashes['CRASH_NUMB']
mass_crashes['MUNI'] = mass_crashes['CITY_TOWN_NAME']
mass_crashes = mass_crashes.drop(columns=["CITY_TOWN_NAME", "CRASH_NUMB", "POLC_AGNCY_TYPE_DESCR"])
mass_crashes['SOURCE'] = 'MASSDOT'

# Merging all Crash data

**All datasets are merged into one dataset that has the following structure:**

**Columns are attributes of each individual crash:**

* SOURCE: specifies where the crash data comes from(MASSDOT, Vision Zero, Somerville
PD and Cambridge PD)
* ID: unique identifier of each crash. Cambridge PD did not have an ID attribute so one
was created by adding CPD_ to the index number of each crash.
* MUNI: specifies the area each crash occurred(Boston, Cambridge, Brookline,
Somerville)
* DATE: the date each crash occurred in this format YEAR-MONTH-DAY
* SEVERITY: it is 1 if the crash was fatal, 2 if the crash resulted in nonfatal injuries and
blank if neither
* CRASH_TIME: time each crash occured
* POLICE: specifies if Local, State, MBTA or Campus police reported the crash
* CYCLIST: 1 if cyclist was involved in the crash, 0 if not
* PEDESTRIAN: 1 if pedestrian was involved in the crash, 0 if not
* OTHER: 1 if other vulnerable user was involved in the crash, 0 if not
* LAT: latitude of the location where the crash occurred
* LON: longitude of the location where the crash occurred
* INTERSTATE: 1 if the crash occurred on an interstate highway, 0 if not

In [5]:
combined = pd.concat([
    mass_crashes
], ignore_index=True)

In [6]:
cols_to_move = [
    "SOURCE",
    
    "ID",

    "MUNI",

    "CRASH_DATE",

    "SEVERITY",

    "CRASH_TIME",

    "POLICE",

    "CYCLIST",

    "PEDESTRIAN",

    "OTHER",

    "LAT",

    "LON",

    "INTERSTATE"
    
]
df_clean = combined[cols_to_move + [c for c in combined.columns if c not in cols_to_move]]
df.head()

,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,CRASH_TIME,CRASH_DATE,CYCLIST,PEDESTRIAN,INTERSTATE,SEVERITY,OTHER,POLICE,ID,MUNI
0,5051236,EVERETT,1.641037e+12,11:00AM to 11:59AM,Closed,Property damage only (none injured),No injury,2,0,0,...,11:34:00,2022-01-01,0,0,0,0.0,0,Local police,5051236,EVERETT
1,5051675,EVERETT,1.641180e+12,03:00AM to 03:59AM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,03:12:00,2022-01-03,0,0,0,2.0,0,Local police,5051675,EVERETT
2,5051674,EVERETT,1.641162e+12,10:00PM to 10:59PM,Closed,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,2,0,...,22:17:00,2022-01-02,0,0,0,2.0,0,Local police,5051674,EVERETT
3,5051661,DEDHAM,1.641127e+12,12:00PM to 12:59PM,Closed,Unknown,Not reported,2,0,0,...,12:36:00,2022-01-02,0,0,0,0.0,0,Local police,5051661,DEDHAM
4,5051578,BRAINTREE,1.641124e+12,11:00AM to 11:59AM,Closed,Non-fatal injury,Possible Injury (C),3,6,0,...,11:49:00,2022-01-02,0,0,0,2.0,0,Local police,5051578,BRAINTREE


In [7]:
# We keep only relevant columns and drop rows that don't have a date
df_short = df.iloc[:, :13]
df_short = df_short.dropna(subset=["CRASH_DATE"])

KeyError: ['CRASH_DATE']

In [ ]:
# We make sure numerical data is of type int or float, and strings of type str
df_short["CRASH_DATE"] = pd.to_datetime(df_short["CRASH_DATE"]).dt.date
df_short['SEVERITY'] = pd.to_numeric(df_short['SEVERITY'], errors='coerce').astype('Int64')
df_short['INTERSTATE'] = df_short['INTERSTATE'].astype(str)
df_short['ID'] = df_short['ID'].astype(str)

In [ ]:
df_short.info()

### Renaming columns to align with Mass Crash Map¶

In [ ]:
renaming = {
    "LAT": "lat",
    "LON": "lng",
    "CRASH_DATE": "date",
    "CRASH_TIME": "time",
    "CYCLIST": "cyclist",
    "PEDESTRIAN": "pedestrian",
    "OTHER": "other",
    "INTERSTATE": "interstate",
    "SEVERITY": "severity",
    "ID": "id",
    "MUNI": "muni",
    "SOURCE": "source",
    "POLICE": "police"
}

df_clean = df_clean.rename(columns= renaming)

In [ ]:
df_clean.to_csv("data/boston-metro-clean.csv")